# ODEs in Python: handing guess-and-check to SymPy

Every equation we have solved since the first week, we solved by recognizing a shape and guessing:
$y' = 6y$, the simple harmonic oscillator $y'' + 9y = 0$, the damped oscillator with a constant push
$y'' + 5y' + 4y = 4$, and Wednesday's board problem $y'' + 5y' + 4y = 130\cos 2x$.

Today we hand all of it to a computer algebra system, SymPy, which does the same guessing faster and
never tells you which rule it used. Two things are worth watching for: where it hands back exactly the
answer you got at the boards, and where it hands back something technically correct and completely useless.

This notebook runs on **jupyterhub.smith.edu** (SymPy 1.14). Run a cell with Shift-Enter.

## 1. Predict first (before you run anything)

1. When SymPy solves the simple harmonic oscillator $y'' + 9y = 0$, will it write the answer with sines and cosines, or with exponentials?
2. Will it hand back the same $13\sin 2x$ that the class got at the boards on Wednesday for $y'' + 5y' + 4y = 130\cos 2x$?
3. Can you write down, in one line, a differential equation that it cannot solve?

Double-click the cell below, type your predictions, then Shift-Enter.

*(your prediction here)*

## 2. Setup

The first line keeps plots inside the notebook on the hub. `init_printing()` turns SymPy's output into typeset mathematics.

In [ ]:
%matplotlib inline
from sympy import *
init_printing()

SymPy is Symbolic Python. A *symbol* is not a Python variable: Python does not know what `x` is until we say so.
Run the next cell and read the error.

In [ ]:
x

Now import the symbols we will use, and try again.

In [ ]:
# These are math symbols, not Python variables with values
from sympy.abc import a, b, c, k, m, n, t, x, y, z

In [ ]:
x

## 3. Derivatives, and checking a guess

Guess-and-check, by machine. The differential equation is
$$\frac{dy}{dx} = 6y.$$
Write a general $y(x)$ that should satisfy it, differentiate it, and test it.

In [ ]:
# My guess
y = a * exp(6*x)
y

In [ ]:
y.diff(x)

In [ ]:
# Does dy/dx equal 6y? The check step is one method call.
y.diff(x).equals(6*y)

Now $$\frac{dy}{dx} = 6x.$$
The guess below has the right shape but the wrong numbers. Change them until the check returns `True`.

In [ ]:
y = 5 * x**3     # wrong on purpose: change the 5 and the 3 until the check below returns True
y

In [ ]:
y.diff(x).equals(6*x)

## 4. `dsolve`: the three lines that replace the whole ritual

For solving, SymPy wants to know that $y$ is an *unknown function* of $x$, not a symbol. We declare it with `Function`.
Then we build the left-hand side of the equation (SymPy assumes the right-hand side is zero), call `dsolve`, and check.

In [ ]:
y = Function('y')
y(x)

In [ ]:
# dy/dx - 6y = 0
ODE_LHS = y(x).diff(x) - 6*y(x)
ODE_LHS

In [ ]:
result = dsolve(ODE_LHS, y(x))
result

In [ ]:
# checkodesol substitutes the result back in; (True, 0) means it works with nothing left over
checkodesol(ODE_LHS, result)

**Why `y = Function('y')` and not `Function('y')(x)`?** With the first form, `y(x)` is the function evaluated at `x`,
so later we can also write `y(0)` for an initial condition. The second form fixes the argument once and for all.

### The ladder we climbed by hand

Run each one, and compare with what you got on paper. First the simple harmonic oscillator, $y'' + 9y = 0$.

In [ ]:
SHO = y(x).diff(x, 2) + 9*y(x)
sho_solution = dsolve(SHO, y(x))
sho_solution

In [ ]:
checkodesol(SHO, sho_solution)

When the right-hand side is not zero, write the whole equation with `Eq`. Class 06's block: $y'' + 5y' + 4y = 4$.

In [ ]:
damped = Eq(y(x).diff(x, 2) + 5*y(x).diff(x) + 4*y(x), 4)
dsolve(damped, y(x))

Wednesday's board problem: $y'' + 5y' + 4y = 130\cos 2x$.

In [ ]:
driven = Eq(y(x).diff(x, 2) + 5*y(x).diff(x) + 4*y(x), 130*cos(2*x))
dsolve(driven, y(x))

## 5. Initial conditions: numbers instead of arbitrary constants

`ics` is a dictionary. `y(0)` sets the value at $x = 0$; `y(x).diff(x).subs(x, 0)` sets the slope there.
The damped oscillator without the push, $y'' + 5y' + 4y = 0$, with $y(0) = 1$ and $y'(0) = 1$:

In [ ]:
homogeneous = y(x).diff(x, 2) + 5*y(x).diff(x) + 4*y(x)
particular = dsolve(homogeneous, y(x), ics={y(0): 1, y(x).diff(x).subs(x, 0): 1})
particular

SymPy factors its answer, which hides both conditions. Check by hand, on paper, that this $y$ really has $y(0) = 1$ and $y'(0) = 1$.
(Multiplying out gives $\tfrac{5}{3}e^{-x} - \tfrac{2}{3}e^{-4x}$.)

## 6. The three things that can go wrong, done by machine

Felder's table (pp. 491-492) lists three places where the plain guess fails. SymPy applies each rescue silently.
Run all three and identify which rescue it used.

In [ ]:
# complex roots: p^2 + 6p + 13 = 0 has p = -3 +/- 2i
dsolve(y(x).diff(x, 2) + 6*y(x).diff(x) + 13*y(x), y(x))

In [ ]:
# double root: p^2 + 6p + 9 = (p + 3)^2
dsolve(y(x).diff(x, 2) + 6*y(x).diff(x) + 9*y(x), y(x))

In [ ]:
# the particular solution collides with a homogeneous one: e^(-2x) is already a solution of the left side
dsolve(Eq(y(x).diff(x, 2) + 3*y(x).diff(x) + 2*y(x), exp(-2*x)), y(x))

Look for the $-x e^{-2x}$ term in the last one. The machine applied the multiply-by-$x$ rescue; the point is that you would not have known to look for it.

## 7. Where it stops helping

Before you run it: what do you expect SymPy to do with
$$\frac{dy}{dx} = \sin(\ln y) + \tan y\,?$$
There is no function whose derivative is the sine of the log of that function plus the tangent of that function.

In [ ]:
dsolve(Eq(y(x).diff(x), sin(log(y(x))) + tan(y(x))), y(x))

No error. It hands back an *implicit* answer with an integral it cannot do: correct, and useless for drawing a curve.

To get a curve out of an equation like this you must supply everything as numbers: every parameter, the initial value, and a domain.
That is a numerical solution, and it is what WHW04's computational problem asks you to build by hand.

## 8. What you found

Go back to your three predictions and answer each one in the cells below.

*(1. sines and cosines, or exponentials? your answer here)*

*(2. did it return 13 sin 2x? your answer here)*

*(3. the equation it could not solve, and what it returned instead)*